# AI-Powered Anomaly Detection for Trade Compliance

This notebook demonstrates machine learning-based anomaly detection on trade compliance data using Isolation Forest. The goal is to identify unusual patterns in customs entries that may indicate errors, fraud, or compliance issues.

**Business Problem**: Trade compliance validation relies on known rules (HTS matching, ADD/CVD flags). But what about **unknown-unknowns** - anomalies that don't violate any explicit rule but still warrant investigation?

**Solution**: Train an unsupervised ML model to learn "normal" trade patterns and flag statistical outliers for human review.

**Key Features**:
- Isolation Forest for unsupervised anomaly detection
- Feature engineering from trade compliance data
- LLM-powered explanation generation for flagged anomalies
- Integration with Snowflake Cortex and ML Registry

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not available - using matplotlib only")

try:
    import snowflake.connector
    SNOWFLAKE_AVAILABLE = True
except ImportError:
    SNOWFLAKE_AVAILABLE = False
    print("snowflake.connector not available - running in demo mode")

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Snowflake available: {SNOWFLAKE_AVAILABLE}")
print(f"Plotly available: {PLOTLY_AVAILABLE}")

In [ ]:
conn = None
if SNOWFLAKE_AVAILABLE:
    try:
        conn = snowflake.connector.connect(
            connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "my_snowflake"
        )
        print("Connected to Snowflake successfully")
    except Exception as e:
        print(f"Could not connect to Snowflake: {e}")
        print("Continuing in demo mode...")

---
## 2. Load Training Data

Load the validation data which contains trade compliance records. This synthetic dataset has **2% injected anomalies** that we'll attempt to detect.

Anomaly types in the data:
- `DUTY_RATIO_OUTLIER`: Unusual duty-to-value ratio
- `UNUSUAL_COO_HTS_COMBO`: Rare country-of-origin and HTS code combination
- `VALUE_SPIKE`: Unusually high entered value
- `BROKER_PATTERN_ANOMALY`: Pattern inconsistent with broker's history

In [ ]:
DATA_PATH = "/Users/rraman/Documents/Cummins_Entry_Visibility/data/synthetic/validation/fact_trade_tariff_validation.csv"

df = pd.read_csv(DATA_PATH)

print(f"Data Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
print("\nColumn Data Types:")
print(df.dtypes)
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
print("\nFirst 5 rows:")
display(df.head())

In [ ]:
print("\n" + "="*60)
print("ANOMALY DISTRIBUTION IN DATASET")
print("="*60)

anomaly_col = 'ANOMALY_FLAG' if 'ANOMALY_FLAG' in df.columns else None

if anomaly_col:
    anomaly_counts = df[anomaly_col].value_counts()
    total = len(df)
    anomaly_pct = (anomaly_counts.get(True, 0) / total) * 100
    
    print(f"\nAnomaly Distribution:")
    print(f"  Normal records:  {anomaly_counts.get(False, 0):,} ({100-anomaly_pct:.1f}%)")
    print(f"  Anomaly records: {anomaly_counts.get(True, 0):,} ({anomaly_pct:.1f}%)")
    
    if 'ANOMALY_TYPE' in df.columns:
        print(f"\nAnomaly Types:")
        anomaly_types = df[df[anomaly_col] == True]['ANOMALY_TYPE'].value_counts()
        for atype, count in anomaly_types.items():
            print(f"  {atype}: {count}")
else:
    print("No ANOMALY_FLAG column found - will treat this as unlabeled data")

---
## 3. Feature Engineering

Create features for anomaly detection from the raw validation data. We'll engineer features across several categories:

1. **Value Features**: ENTERED_VALUE, log-transformed value, percentiles by HTS
2. **Category Encodings**: Broker, Country of Origin encoded as numbers
3. **HTS-Based Features**: Chapter (first 2 digits), Section groupings
4. **Validation Flags**: Binary indicators for each validation check (part_fail, hts_fail, etc.)

In [ ]:
def extract_hts_chapter(hts_code):
    """Extract 2-digit chapter from HTS code."""
    if pd.isna(hts_code):
        return 0
    hts_str = str(hts_code).replace('.', '').replace('-', '')
    return int(hts_str[:2]) if len(hts_str) >= 2 and hts_str[:2].isdigit() else 0

def hts_chapter_to_section(chapter):
    """Map HTS chapter to section (1-22)."""
    section_ranges = [
        (1, 5, 1),    # Live animals, animal products
        (6, 14, 2),   # Vegetable products
        (15, 15, 3),  # Animal/vegetable fats
        (16, 24, 4),  # Prepared foodstuffs
        (25, 27, 5),  # Mineral products
        (28, 38, 6),  # Chemical products
        (39, 40, 7),  # Plastics and rubber
        (41, 43, 8),  # Leather and skins
        (44, 46, 9),  # Wood products
        (47, 49, 10), # Paper products
        (50, 63, 11), # Textiles
        (64, 67, 12), # Footwear, headgear
        (68, 70, 13), # Stone, glass products
        (71, 71, 14), # Precious metals/stones
        (72, 83, 15), # Base metals
        (84, 85, 16), # Machinery, electrical
        (86, 89, 17), # Vehicles, aircraft
        (90, 92, 18), # Instruments
        (93, 93, 19), # Arms and ammunition
        (94, 96, 20), # Miscellaneous manufactured
        (97, 97, 21), # Works of art
        (98, 99, 22), # Special classification
    ]
    for low, high, section in section_ranges:
        if low <= chapter <= high:
            return section
    return 0

In [ ]:
def engineer_features(df):
    """Create features for anomaly detection."""
    
    features = pd.DataFrame(index=df.index)
    
    if 'ENTERED_VALUE' in df.columns:
        features['entered_value'] = df['ENTERED_VALUE'].fillna(0)
        features['log_value'] = np.log1p(features['entered_value'])
    elif 'LINE_NUMBER' in df.columns:
        np.random.seed(42)
        features['entered_value'] = np.random.lognormal(mean=8, sigma=2, size=len(df))
        features['log_value'] = np.log1p(features['entered_value'])
    else:
        features['entered_value'] = 0
        features['log_value'] = 0
    
    if 'DUTY_AMOUNT' in df.columns:
        features['duty_amount'] = df['DUTY_AMOUNT'].fillna(0)
        features['duty_ratio'] = features['duty_amount'] / (features['entered_value'] + 1)
    else:
        np.random.seed(43)
        features['duty_amount'] = features['entered_value'] * np.random.uniform(0.01, 0.25, len(df))
        features['duty_ratio'] = features['duty_amount'] / (features['entered_value'] + 1)
    
    broker_col = None
    for col in ['BROKER_NAME', 'BROKER', 'BROKER_ID']:
        if col in df.columns:
            broker_col = col
            break
    
    if broker_col:
        le_broker = LabelEncoder()
        features['broker_encoded'] = le_broker.fit_transform(df[broker_col].fillna('UNKNOWN'))
    else:
        features['broker_encoded'] = 0
    
    coo_col = None
    for col in ['COUNTRY_OF_ORIGIN', 'COO', 'ORIGIN_COUNTRY']:
        if col in df.columns:
            coo_col = col
            break
    
    if coo_col:
        le_coo = LabelEncoder()
        features['coo_encoded'] = le_coo.fit_transform(df[coo_col].fillna('XX'))
    else:
        np.random.seed(44)
        features['coo_encoded'] = np.random.randint(0, 50, len(df))
    
    hts_col = None
    for col in ['GTM_HS_CODE', 'HTS_CODE', 'HS_CODE', 'TARIFF_CODE']:
        if col in df.columns:
            hts_col = col
            break
    
    if hts_col:
        features['hts_chapter'] = df[hts_col].apply(extract_hts_chapter)
        features['hts_section'] = features['hts_chapter'].apply(hts_chapter_to_section)
    else:
        np.random.seed(45)
        features['hts_chapter'] = np.random.randint(1, 99, len(df))
        features['hts_section'] = features['hts_chapter'].apply(hts_chapter_to_section)
    
    for status_col in ['PART_NUMBER_STATUS', 'HS_CODE_STATUS', 'ADD_CVD_STATUS', 'PREFERENTIAL_PROGRAM_STATUS']:
        if status_col in df.columns:
            feature_name = status_col.lower().replace('_status', '_fail')
            features[feature_name] = (df[status_col] == 'FAIL').astype(int)
    
    features['overall_fail'] = (df['OVERALL_STATUS'] == 'FAIL').astype(int) if 'OVERALL_STATUS' in df.columns else 0
    
    if 'ANOMALY_SCORE' in df.columns:
        features['injected_anomaly_score'] = df['ANOMALY_SCORE'].fillna(0)
    
    features['line_number'] = df['LINE_NUMBER'].fillna(1) if 'LINE_NUMBER' in df.columns else 1
    
    return features

print("Engineering features...")
features_df = engineer_features(df)
print(f"\nFeatures created: {list(features_df.columns)}")
print(f"Feature matrix shape: {features_df.shape}")

In [ ]:
print("\nFeature Statistics:")
display(features_df.describe())

In [ ]:
print("\nHandling missing values...")
print(f"Missing values before: {features_df.isnull().sum().sum()}")
features_df = features_df.fillna(0)
print(f"Missing values after: {features_df.isnull().sum().sum()}")

In [ ]:
model_features = [
    'log_value', 'duty_ratio', 'broker_encoded', 'coo_encoded',
    'hts_chapter', 'hts_section', 'line_number'
]

for col in ['part_number_fail', 'hs_code_fail', 'add_cvd_fail', 'preferential_program_fail', 'overall_fail']:
    if col in features_df.columns:
        model_features.append(col)

X = features_df[model_features].copy()

print(f"\nFeatures for model training: {model_features}")
print(f"Training matrix shape: {X.shape}")

In [ ]:
print("\nNormalizing features with StandardScaler...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=model_features, index=X.index)
print(f"\nScaled feature statistics:")
display(X_scaled_df.describe().round(3))

---
## 4. Train Isolation Forest

**Isolation Forest** is an unsupervised anomaly detection algorithm that works by:
1. Randomly selecting a feature
2. Randomly selecting a split value between min/max of that feature
3. Recursively partitioning until each point is isolated

**Key insight**: Anomalies are easier to isolate (require fewer splits), so they have shorter average path lengths.

We set `contamination=0.02` since we know ~2% of our data contains injected anomalies.

In [ ]:
CONTAMINATION = 0.02
RANDOM_STATE = 42
N_ESTIMATORS = 100

print("Training Isolation Forest...")
print(f"  n_estimators: {N_ESTIMATORS}")
print(f"  contamination: {CONTAMINATION} ({CONTAMINATION*100}%)")
print(f"  random_state: {RANDOM_STATE}")

iso_forest = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

iso_forest.fit(X_scaled)
print("\nModel training complete!")

In [ ]:
print("\nGenerating anomaly scores...")

anomaly_scores = iso_forest.decision_function(X_scaled)

anomaly_predictions = iso_forest.predict(X_scaled)

features_df['anomaly_score'] = anomaly_scores
features_df['is_anomaly'] = (anomaly_predictions == -1).astype(int)

print(f"\nAnomaly Score Statistics:")
print(f"  Min: {anomaly_scores.min():.4f}")
print(f"  Max: {anomaly_scores.max():.4f}")
print(f"  Mean: {anomaly_scores.mean():.4f}")
print(f"  Std: {anomaly_scores.std():.4f}")

print(f"\nPredictions:")
print(f"  Normal: {(features_df['is_anomaly'] == 0).sum():,}")
print(f"  Anomaly: {(features_df['is_anomaly'] == 1).sum():,}")

In [ ]:
threshold_default = features_df.loc[features_df['is_anomaly'] == 1, 'anomaly_score'].max()
print(f"\nDefault threshold (contamination={CONTAMINATION}): {threshold_default:.4f}")
print("Records with anomaly_score below this threshold are flagged as anomalies.")

---
## 5. Evaluate Detection

Since we have ground truth labels (the injected 2% anomalies), we can evaluate how well the model detected them.

In [ ]:
if 'ANOMALY_FLAG' in df.columns:
    y_true = df['ANOMALY_FLAG'].astype(int).values
    y_pred = features_df['is_anomaly'].values
    
    print("\n" + "="*60)
    print("MODEL EVALUATION (Ground Truth Available)")
    print("="*60)
    
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly']))
    
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\nSummary Metrics:")
    print(f"  Precision: {precision:.3f} (of flagged anomalies, what % were real?)")
    print(f"  Recall:    {recall:.3f} (of real anomalies, what % did we catch?)")
    print(f"  F1 Score:  {f1:.3f} (harmonic mean of precision and recall)")
else:
    print("No ground truth labels available - skipping evaluation metrics")
    y_true = None

In [ ]:
if y_true is not None:
    cm = confusion_matrix(y_true, y_pred)
    
    print("\nConfusion Matrix:")
    print("                 Predicted")
    print("                 Normal  Anomaly")
    print(f"Actual Normal    {cm[0,0]:6}  {cm[0,1]:6}")
    print(f"       Anomaly   {cm[1,0]:6}  {cm[1,1]:6}")
    
    tn, fp, fn, tp = cm.ravel()
    print(f"\nBreakdown:")
    print(f"  True Negatives (correctly identified normal):  {tn:,}")
    print(f"  False Positives (normal flagged as anomaly):   {fp:,}")
    print(f"  False Negatives (anomaly missed):              {fn:,}")
    print(f"  True Positives (correctly caught anomaly):     {tp:,}")

In [ ]:
if y_true is not None:
    print("\n" + "="*60)
    print("THRESHOLD ANALYSIS")
    print("="*60)
    print("\nEvaluating different thresholds for anomaly flagging:")
    print(f"{'Percentile':>12} {'Threshold':>12} {'Precision':>12} {'Recall':>12} {'F1':>12}")
    print("-" * 64)
    
    for pct in [1, 2, 3, 5, 10]:
        threshold = np.percentile(anomaly_scores, pct)
        y_pred_thresh = (anomaly_scores < threshold).astype(int)
        
        prec = precision_score(y_true, y_pred_thresh, zero_division=0)
        rec = recall_score(y_true, y_pred_thresh, zero_division=0)
        f1_thresh = f1_score(y_true, y_pred_thresh, zero_division=0)
        
        print(f"{pct:>11}% {threshold:>12.4f} {prec:>12.3f} {rec:>12.3f} {f1_thresh:>12.3f}")

---
## 6. LLM Explanation Layer

For detected anomalies, we can use Snowflake Cortex COMPLETE to generate human-readable explanations. This helps compliance analysts understand **why** a record was flagged and **what action** to take.

In [ ]:
def get_feature_contribution(row, scaler, feature_names, model):
    """Calculate which features contributed most to the anomaly score."""
    
    mean_vals = scaler.mean_
    std_vals = scaler.scale_
    
    contributions = {}
    for i, feature in enumerate(feature_names):
        z_score = (row[feature] - mean_vals[i]) / std_vals[i] if std_vals[i] > 0 else 0
        contributions[feature] = abs(z_score)
    
    sorted_contributions = sorted(contributions.items(), key=lambda x: x[1], reverse=True)
    return sorted_contributions[:3]


def build_explanation_prompt(record_dict, top_contributors, anomaly_type=None):
    """Build a prompt for LLM to explain the anomaly."""
    
    prompt = f"""You are a trade compliance expert. Analyze this flagged customs entry and provide a brief explanation.

FLAGGED RECORD:
- Entry Number: {record_dict.get('ENTRY_NUMBER', 'N/A')}
- Part Number: {record_dict.get('PART_NUMBER', 'N/A')}
- HTS Code: {record_dict.get('GTM_HS_CODE', record_dict.get('HTS_CODE', 'N/A'))}
- Country of Origin: {record_dict.get('COUNTRY_OF_ORIGIN', 'N/A')}
- Entered Value: ${record_dict.get('entered_value', 0):,.2f}
- Duty Amount: ${record_dict.get('duty_amount', 0):,.2f}
- Broker: {record_dict.get('BROKER_NAME', 'N/A')}

TOP UNUSUAL FEATURES:
1. {top_contributors[0][0]}: z-score = {top_contributors[0][1]:.2f}
2. {top_contributors[1][0]}: z-score = {top_contributors[1][1]:.2f}
3. {top_contributors[2][0]}: z-score = {top_contributors[2][1]:.2f}

ANOMALY TYPE: {anomaly_type or 'Statistical outlier'}

Provide a 2-3 sentence explanation of why this record was flagged and recommend one specific action.

Format your response as:
EXPLANATION: [your explanation]
RECOMMENDED ACTION: [your action]
"""
    return prompt

In [ ]:
def generate_explanation_via_cortex(conn, prompt):
    """Call Snowflake Cortex COMPLETE for LLM-generated explanation."""
    
    if conn is None:
        return None
    
    try:
        cursor = conn.cursor()
        escaped_prompt = prompt.replace("'", "''")
        
        sql = f"""
            SELECT SNOWFLAKE.CORTEX.COMPLETE(
                'mistral-large',
                '{escaped_prompt}'
            ) AS explanation
        """
        
        cursor.execute(sql)
        result = cursor.fetchone()[0]
        cursor.close()
        return result
        
    except Exception as e:
        print(f"Cortex call failed: {e}")
        return None


def simulate_explanation(record_dict, top_contributors, anomaly_type=None):
    """Generate simulated explanation for demo mode."""
    
    explanations = {
        'DUTY_RATIO_OUTLIER': {
            'explanation': f"This entry shows an unusual duty-to-value ratio of {record_dict.get('duty_ratio', 0):.2%}, which is significantly different from typical entries for this HTS code. This could indicate incorrect duty calculation or misclassification.",
            'action': "Review duty calculation with broker and verify HTS classification against product specifications."
        },
        'UNUSUAL_COO_HTS_COMBO': {
            'explanation': f"This entry combines Country of Origin with HTS chapter {record_dict.get('hts_chapter', 'N/A')} in a pattern rarely seen in historical data. This combination may warrant ADD/CVD review.",
            'action': "Verify country of origin documentation and check ADD/CVD applicability for this product/country combination."
        },
        'VALUE_SPIKE': {
            'explanation': f"The entered value of ${record_dict.get('entered_value', 0):,.2f} is significantly higher than typical values for this part/HTS combination. This could indicate a data entry error or pricing anomaly.",
            'action': "Request commercial invoice review and verify unit pricing with supplier."
        },
        'default': {
            'explanation': f"This entry was flagged due to unusual patterns in {top_contributors[0][0]} and {top_contributors[1][0]}. The combination of values differs statistically from normal trade patterns.",
            'action': "Review entry documentation and compare against similar recent entries from this broker."
        }
    }
    
    result = explanations.get(anomaly_type, explanations['default'])
    return f"EXPLANATION: {result['explanation']}\nRECOMMENDED ACTION: {result['action']}"

In [ ]:
anomaly_indices = features_df[features_df['is_anomaly'] == 1].nsmallest(5, 'anomaly_score').index

print("\n" + "="*70)
print("LLM EXPLANATIONS FOR TOP ANOMALIES")
print("="*70)

for idx in anomaly_indices:
    record = df.loc[idx].to_dict()
    feature_row = features_df.loc[idx]
    
    for key in feature_row.index:
        record[key] = feature_row[key]
    
    X_row = X.loc[idx]
    top_contributors = get_feature_contribution(X_row, scaler, model_features, iso_forest)
    
    anomaly_type = record.get('ANOMALY_TYPE', None)
    if pd.isna(anomaly_type) or anomaly_type == '':
        anomaly_type = None
    
    prompt = build_explanation_prompt(record, top_contributors, anomaly_type)
    
    explanation = generate_explanation_via_cortex(conn, prompt)
    
    if explanation is None:
        explanation = simulate_explanation(record, top_contributors, anomaly_type)
    
    print(f"\n{'-'*70}")
    print(f"ANOMALY #{idx} (Score: {feature_row['anomaly_score']:.4f})")
    print(f"Entry: {record.get('ENTRY_NUMBER', 'N/A')} | Part: {record.get('PART_NUMBER', 'N/A')}")
    print(f"Type: {anomaly_type or 'Statistical outlier'}")
    print(f"\n{explanation}")

In [ ]:
print("\n" + "="*70)
print("EXAMPLE: Cortex COMPLETE SQL Call")
print("="*70)

example_sql = """
-- Generate explanation for anomaly using Snowflake Cortex
SELECT 
    entry_number,
    part_number,
    anomaly_score,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large',
        'You are a trade compliance expert. This customs entry was flagged as an anomaly: ' ||
        'Entry: ' || entry_number || ', Part: ' || part_number || ', Value: $' || entered_value ||
        ', Duty Ratio: ' || duty_ratio || '. Explain why this might be unusual and recommend an action.' ||
        ' Keep response under 100 words.'
    ) AS llm_explanation
FROM TRADE_COMPLIANCE_DB.ML.ANOMALY_DETECTIONS
WHERE is_anomaly = TRUE
ORDER BY anomaly_score ASC
LIMIT 10;
"""
print(example_sql)

---
## 7. Visualize Results

Visualize the anomaly detection results using:
1. t-SNE dimensionality reduction to see clustering
2. Anomaly score distribution
3. Top anomalies by score

In [ ]:
print("\nComputing t-SNE embedding (this may take a minute)...")

sample_size = min(2000, len(X_scaled))
np.random.seed(42)
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)

X_sample = X_scaled[sample_indices]
labels_sample = features_df['is_anomaly'].iloc[sample_indices].values
scores_sample = features_df['anomaly_score'].iloc[sample_indices].values

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_sample)

print(f"t-SNE complete. Sampled {sample_size} points.")

In [ ]:
if PLOTLY_AVAILABLE:
    tsne_df = pd.DataFrame({
        'x': X_tsne[:, 0],
        'y': X_tsne[:, 1],
        'anomaly': ['Anomaly' if a == 1 else 'Normal' for a in labels_sample],
        'score': scores_sample
    })
    
    fig = px.scatter(
        tsne_df, x='x', y='y', color='anomaly',
        color_discrete_map={'Normal': 'blue', 'Anomaly': 'red'},
        title='t-SNE Visualization of Trade Compliance Data',
        labels={'x': 't-SNE Dimension 1', 'y': 't-SNE Dimension 2'},
        opacity=0.6
    )
    fig.update_layout(height=600)
    fig.show()
else:
    plt.figure(figsize=(12, 8))
    normal_mask = labels_sample == 0
    anomaly_mask = labels_sample == 1
    
    plt.scatter(X_tsne[normal_mask, 0], X_tsne[normal_mask, 1], 
                c='blue', alpha=0.5, label='Normal', s=20)
    plt.scatter(X_tsne[anomaly_mask, 0], X_tsne[anomaly_mask, 1], 
                c='red', alpha=0.8, label='Anomaly', s=50, marker='x')
    
    plt.title('t-SNE Visualization of Trade Compliance Data')
    plt.xlabel('t-SNE Dimension 1')
    plt.ylabel('t-SNE Dimension 2')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if PLOTLY_AVAILABLE:
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        'Anomaly Score Distribution',
        'Score Distribution by Class'
    ))
    
    fig.add_trace(
        go.Histogram(x=features_df['anomaly_score'], nbinsx=50, name='All Records'),
        row=1, col=1
    )
    
    normal_scores = features_df[features_df['is_anomaly'] == 0]['anomaly_score']
    anomaly_scores_detected = features_df[features_df['is_anomaly'] == 1]['anomaly_score']
    
    fig.add_trace(
        go.Histogram(x=normal_scores, nbinsx=30, name='Normal', opacity=0.7),
        row=1, col=2
    )
    fig.add_trace(
        go.Histogram(x=anomaly_scores_detected, nbinsx=30, name='Anomaly', opacity=0.7),
        row=1, col=2
    )
    
    fig.update_layout(height=400, title_text='Anomaly Score Distributions', barmode='overlay')
    fig.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(features_df['anomaly_score'], bins=50, color='steelblue', edgecolor='black')
    axes[0].set_title('Anomaly Score Distribution')
    axes[0].set_xlabel('Anomaly Score')
    axes[0].set_ylabel('Count')
    
    normal_scores = features_df[features_df['is_anomaly'] == 0]['anomaly_score']
    anomaly_scores_detected = features_df[features_df['is_anomaly'] == 1]['anomaly_score']
    
    axes[1].hist(normal_scores, bins=30, alpha=0.7, label='Normal', color='blue')
    axes[1].hist(anomaly_scores_detected, bins=30, alpha=0.7, label='Anomaly', color='red')
    axes[1].set_title('Score Distribution by Class')
    axes[1].set_xlabel('Anomaly Score')
    axes[1].set_ylabel('Count')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
print("\n" + "="*70)
print("TOP 10 ANOMALIES BY SCORE")
print("="*70)

top_anomalies = features_df[features_df['is_anomaly'] == 1].nsmallest(10, 'anomaly_score')

display_cols = ['anomaly_score', 'log_value', 'duty_ratio', 'hts_chapter', 'coo_encoded']
if 'overall_fail' in features_df.columns:
    display_cols.append('overall_fail')

top_anomalies_display = top_anomalies[display_cols].copy()

for idx in top_anomalies.index:
    if 'ANOMALY_TYPE' in df.columns:
        top_anomalies_display.loc[idx, 'anomaly_type'] = df.loc[idx, 'ANOMALY_TYPE']

display(top_anomalies_display)

---
## 8. Save Model & Results

Save the trained model and anomaly detections for production use.

In [ ]:
print("\n" + "="*70)
print("SAVING ANOMALY DETECTIONS")
print("="*70)

results_df = df.copy()
results_df['ML_ANOMALY_SCORE'] = features_df['anomaly_score']
results_df['ML_IS_ANOMALY'] = features_df['is_anomaly']
results_df['DETECTION_TIMESTAMP'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
results_df['MODEL_VERSION'] = 'isolation_forest_v1'

output_path = "/Users/rraman/Documents/Cummins_Entry_Visibility/data/synthetic/validation/anomaly_detections.csv"
results_df.to_csv(output_path, index=False)
print(f"\nResults saved to: {output_path}")
print(f"Total records: {len(results_df):,}")
print(f"Anomalies flagged: {results_df['ML_IS_ANOMALY'].sum():,}")

In [ ]:
print("\n" + "="*70)
print("SNOWFLAKE ML.ANOMALY_DETECTIONS TABLE DDL")
print("="*70)

ddl = """
-- Create table to store anomaly detection results
CREATE TABLE IF NOT EXISTS TRADE_COMPLIANCE_DB.ML.ANOMALY_DETECTIONS (
    DETECTION_ID VARCHAR(36) DEFAULT UUID_STRING(),
    VALIDATION_ID VARCHAR(36),
    LINE_ID VARCHAR(36),
    ENTRY_NUMBER VARCHAR(20),
    ENTRY_DATE DATE,
    BROKER_NAME VARCHAR(50),
    PART_NUMBER VARCHAR(50),
    HTS_CODE VARCHAR(20),
    COUNTRY_OF_ORIGIN VARCHAR(3),
    ENTERED_VALUE NUMBER(18,2),
    DUTY_AMOUNT NUMBER(18,2),
    
    -- ML Model Outputs
    ML_ANOMALY_SCORE FLOAT,
    ML_IS_ANOMALY BOOLEAN,
    ML_MODEL_VERSION VARCHAR(50),
    
    -- Feature Values
    FEATURE_LOG_VALUE FLOAT,
    FEATURE_DUTY_RATIO FLOAT,
    FEATURE_HTS_CHAPTER INTEGER,
    
    -- LLM Explanation
    LLM_EXPLANATION VARCHAR(2000),
    RECOMMENDED_ACTION VARCHAR(500),
    
    -- Metadata
    DETECTION_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    REVIEW_STATUS VARCHAR(20) DEFAULT 'PENDING',
    REVIEWED_BY VARCHAR(100),
    REVIEW_TIMESTAMP TIMESTAMP_NTZ,
    REVIEW_NOTES VARCHAR(1000)
);

-- Create view for pending reviews
CREATE OR REPLACE VIEW TRADE_COMPLIANCE_DB.ML.V_ANOMALIES_PENDING_REVIEW AS
SELECT *
FROM TRADE_COMPLIANCE_DB.ML.ANOMALY_DETECTIONS
WHERE ML_IS_ANOMALY = TRUE
  AND REVIEW_STATUS = 'PENDING'
ORDER BY ML_ANOMALY_SCORE ASC;
"""
print(ddl)

In [ ]:
print("\n" + "="*70)
print("SNOWFLAKE ML REGISTRY - MODEL REGISTRATION")
print("="*70)

registry_code = '''
# To register this model in Snowflake ML Registry:

from snowflake.ml.registry import Registry
from snowflake.snowpark import Session

# Create session
session = Session.builder.configs({
    "connection_name": "my_snowflake"
}).create()

# Initialize registry
registry = Registry(session=session, database_name="TRADE_COMPLIANCE_DB", schema_name="ML")

# Log the model
model_version = registry.log_model(
    model=iso_forest,
    model_name="ANOMALY_DETECTOR",
    version_name="v1",
    comment="Isolation Forest for trade compliance anomaly detection. Contamination=0.02",
    metrics={
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "contamination": CONTAMINATION,
        "n_estimators": N_ESTIMATORS
    },
    sample_input_data=X_scaled_df.head(10),
    conda_dependencies=["scikit-learn"]
)

print(f"Model registered: {model_version.fully_qualified_model_name}")

# Deploy for inference
model_version.create_service(
    service_name="ANOMALY_DETECTOR_SERVICE",
    compute_pool="ML_COMPUTE_POOL"
)
'''
print(registry_code)

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"""
ANOMALY DETECTION MODEL SUMMARY
{'='*50}

Training Data:
  - Records: {len(df):,}
  - Features: {len(model_features)}
  - Injected anomalies: {df['ANOMALY_FLAG'].sum() if 'ANOMALY_FLAG' in df.columns else 'N/A'}

Model Configuration:
  - Algorithm: Isolation Forest
  - Estimators: {N_ESTIMATORS}
  - Contamination: {CONTAMINATION} ({CONTAMINATION*100}%)

Results:
  - Anomalies detected: {features_df['is_anomaly'].sum():,}
  - Precision: {precision:.3f if y_true is not None else 'N/A'}
  - Recall: {recall:.3f if y_true is not None else 'N/A'}
  - F1 Score: {f1:.3f if y_true is not None else 'N/A'}

Output Files:
  - Detections: {output_path}

Next Steps:
  1. Review flagged anomalies with compliance team
  2. Register model in Snowflake ML Registry
  3. Set up scheduled scoring pipeline
  4. Collect feedback to improve model
""")

In [ ]:
if conn:
    conn.close()
    print("Snowflake connection closed.")